In [3]:
# Cell 1: Imports and Setup
import numpy as np
import pandas as pd
import time
import os
import sqlite3
from statistics import mean, stdev
from joblib import Parallel, delayed
from IPython.display import display

# Import your project files
from algo import metric_list
from foldrm import Classifier
from utils import split_data, split_xy, get_scores, count_rules_in_model, num_predicates, get_inverse_brier_score
from datasets import wine, cars, ecoli, weight_lifting, wall_robot, page_blocks, nursery, dry_bean, acute, autism,breastw,credit,flags,glass,heart,ionosphere,kidney,voting,sonar


# --- Experiment Configuration ---
datasets = [acute, wine, cars, glass, flags, ecoli, sonar, weight_lifting] #ecoli, weight_lifting, acute, autism,breastw,credit,flags,glass,heart,ionosphere,kidney,voting,sonar #wall_robot, page_blocks, nursery, dry_bean,
dataset_names = ["acute", "Wine", "cars", "glass", "flags", "Ecoli","sonar","Weight Lifting"] # #, "Ecoli", "Weight Lifting", "acute", "autism", "breastw", "credit" , "flags", "glass", "heart", "ionosphere" , "kidney", "voting","sonar", "Wall Robot", "Page Blocks"   #"Wine", "Ecoli", "Weight Lifting", "Wall Robot", "Page Blocks", "Nursery", "Dry Bean"

# Define the fit methods and strategies to test
fit_methods = ["FOLD-RM (fit)"] #, "CON-FOLD (confidence_fit)"
selection_strategies = ['greedy', 'round_robin', 'best_rule', 'info_gain']

# We will fix the gain metric to the default to isolate the effect of the selection strategy
#FIXED_METRIC = 'information_gain' 
metric_list = ['original', 'original_guarded', 'original_reweighted', 'original_t_removed', 'IG_over_P', 'IG_over_P2', 'LogOddsRatio',  'MathewsCC',
               'MathewsCC_mod', 'MathewsP', 'Gini_Impurity', 'Precision_Gini_Impurity']

NUM_TRIALS = 30 # Reduced for quicker testing; you can set this back to 300
DB_FILE = 'class_selection_experiment_4_30trials.db'

# Display options for the final DataFrame
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Cell 2: Helper Function for a Single Trial
def run_one_trial(dataset_name, strategy, fit_method, metric, data, model_template, num_classes, db_file):
    """
    Runs a single trial for a given selection strategy and fit method, and saves results to SQLite.
    """
    start_time = time.time()
    
    model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
    data_train, data_test = split_data(data, ratio=0.8)
    X_test, Y_test = split_xy(data_test)
    
    # Conditionally call the correct fit method
    if fit_method == "CON-FOLD (confidence_fit)":
        model.confidence_fit(data_train, metric=metric, num_classes=num_classes, selection_strategy=strategy)
    else: # FOLD-RM (fit)
        model.fit(data_train, metric=metric, num_classes=num_classes, selection_strategy=strategy)
    
    Ystar_test_tuples = model.predict(X_test)
    Ystar_test = [y[0] for y in Ystar_test_tuples]
    score = get_scores(Ystar_test, data_test)
    brier_score = get_inverse_brier_score(Ystar_test_tuples, Y_test)
    
    rule_count = count_rules_in_model(model)
    
    model.asp()
    predicate_count = num_predicates(model)
    
    elapsed_time = time.time() - start_time
    
    # Save to database, now including fit_method
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute('PRAGMA journal_mode=WAL;')
    cursor.execute('''
        INSERT INTO trials (dataset, strategy, fit_method, metric, accuracy, brier_score, time, num_rules, num_preds)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (dataset_name, strategy, fit_method, metric, score, brier_score, elapsed_time, rule_count, predicate_count))
    conn.commit()
    conn.close()

def run_experiments_for_combination(dataset_info, strategy, fit_method, metric, remaining_trials, db_file):
    """
    Runs the remaining trials for a specific dataset, strategy, and fit method in parallel.
    """
    dataset_func, name = dataset_info
    print(f"--- Starting: Dataset='{name}', Strategy='{strategy}', Fit Method='{fit_method}', 'Metric={metric}', Remaining Trials={remaining_trials} ---")
    
    model_template, data = dataset_func()
    num_classes = len(pd.unique(pd.DataFrame(data).iloc[:, -1]))
    
    if remaining_trials > 0:
        Parallel(n_jobs=-1, verbose=5)(
            delayed(run_one_trial)(name, strategy, fit_method, metric, data, model_template, num_classes, db_file)
            for _ in range(remaining_trials)
        )
    
    print(f"--- Finished: Dataset='{name}', Strategy='{strategy}', Fit Method='{fit_method}' ---")

# Cell 3: Main Experiment Runner with Smart Resume
if __name__ == "__main__":
    datasets_with_names = list(zip(datasets, dataset_names))

    # Create database and table if not exists
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    # Add the fit_method column to the schema
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS trials (
            dataset TEXT,
            strategy TEXT,
            fit_method TEXT,
            metric TEXT,
            accuracy REAL,
            brier_score REAL,
            time REAL,
            num_rules INTEGER,
            num_preds INTEGER
        )
    ''')
    conn.commit()

    # Get current counts for each combination
    counts = {}
    cursor.execute('SELECT dataset, strategy, fit_method, metric, COUNT(*) FROM trials GROUP BY dataset, strategy, fit_method, metric')
    for row in cursor.fetchall():
        ds, strat, fm, met, cnt = row
        counts[(ds, strat, fm, met)] = cnt
    conn.close()

    # Create list of tasks to run, now iterating over fit_methods
    tasks_to_run = []
    for dataset_info in datasets_with_names:
        name = dataset_info[1]
        for strategy in selection_strategies:
            for fm in fit_methods:
                for metric in metric_list:
                    key = (name, strategy, fm, metric)
                    current_trials = counts.get(key, 0)
                    remaining = NUM_TRIALS - current_trials
                    if remaining > 0:
                        tasks_to_run.append({
                            'dataset_info': dataset_info,
                            'strategy': strategy,
                            'fit_method': fm,
                            'metric': metric,
                            'remaining_trials': remaining,
                            'db_file': DB_FILE
                        })

    print(f"\n>>> Found {len(tasks_to_run)} combinations with remaining trials. <<<\n")

    # Run tasks
    if tasks_to_run:
        for task in tasks_to_run:
            run_experiments_for_combination(**task)
        print("\n--- All tasks complete! You can now run the aggregation cell. ---")
    else:
        print("\n--- No remaining trials to run. Everything is complete. ---")

# Cell 4: Aggregate, Display, and Save Final Results
def process_and_display_results(df, fit_method_name, filename):
    """
    Helper function to pivot, average, format, display, and save results for a single fit method.
    """
    if df.empty:
        print(f"No completed data to process for {fit_method_name}.")
        return

    # 1. Group by dataset and strategy to get aggregates
    agg_df = df.groupby(['dataset', 'strategy']).agg({
        'accuracy': ['mean', 'std'], 'brier_score': ['mean', 'std'],
        'time': ['mean', 'std'], 'num_rules': ['mean', 'std'], 'num_preds': ['mean', 'std']
    })
    agg_df.columns = ['_'.join(col).strip() for col in agg_df.columns.values]
    agg_df.reset_index(inplace=True)

    # 2. Pivot the table for a nice comparison view
    pivoted = agg_df.pivot(index='dataset', columns='strategy')
    
    # 3. Reorder columns to group by metric (Accuracy, Brier, Time, etc.)
    metric_order = ['accuracy_mean', 'accuracy_std', 'brier_score_mean', 'brier_score_std', 
                    'time_mean', 'num_rules_mean', 'num_preds_mean']
    strategy_order = selection_strategies
    
    final_cols = [(metric, strategy) for metric in metric_order for strategy in strategy_order if (metric, strategy) in pivoted.columns]
    pivoted = pivoted.reindex(columns=final_cols)

    # 4. Add an 'Average' row across all datasets
    avg_row = agg_df.groupby('strategy').mean(numeric_only=True)
    avg_row.name = 'Average (All Datasets)'
    avg_row_pivoted = avg_row.unstack().to_frame().T
    avg_row_pivoted.index = [avg_row.name]
    avg_row_pivoted.columns = pd.MultiIndex.from_tuples(avg_row_pivoted.columns)
    
    final_summary = pd.concat([pivoted, avg_row_pivoted])

    # 5. Display and Save
    print(f"\n{'='*40}")
    print(f"--- STRATEGY EXPERIMENT SUMMARY: {fit_method_name} ---")
    
    styled_df = final_summary.style.format("{:.4f}", na_rep="-") \
        .background_gradient(cmap='viridis', axis=1, 
                             subset=[(m, s) for m in ['accuracy_mean', 'brier_score_mean'] for s in strategy_order])

    display(styled_df)
    
    #final_summary.to_csv(filename)
    #print(f"\nSummary results for {fit_method_name} saved to {filename}")
    print(f"{'='*40}")


if __name__ == "__main__":
    backup_file = 'strategy_experiment_4_30trials_results_full.csv'

    # 1. Load all trial data from the database
    conn = sqlite3.connect(DB_FILE)
    try:
        all_trials_df = pd.read_sql_query("SELECT * FROM trials", conn)
    except pd.io.sql.DatabaseError:
        all_trials_df = pd.DataFrame()
    conn.close()

    if not all_trials_df.empty:
        # 2. Get trial counts and filter for completed experiments
        trial_counts = all_trials_df.groupby(['dataset', 'strategy', 'fit_method', 'metric']).size().reset_index(name='trial_count')
        completed_combinations = trial_counts[trial_counts['trial_count'] >= NUM_TRIALS]
        
        # Merge back to get only the data from completed runs
        completed_df = pd.merge(all_trials_df, completed_combinations, on=['dataset', 'strategy', 'fit_method', 'metric'])

        if not completed_df.empty:
            # Save a backup of all completed trial data
            completed_df.to_csv(backup_file, index=False)
            print(f"Backup of all completed trial data saved to {backup_file}")
        
            # Get the list of metrics that have completed trials
            metrics_to_process = completed_df['metric'].unique()
        
            # Loop through each metric and generate a separate report
            for metric in metrics_to_process:
                print(f"\n\n{'#'*80}")
                print(f"### PROCESSING RESULTS FOR METRIC: {metric} ###")
                print(f"{'#'*80}")
                
                metric_df = completed_df[completed_df['metric'] == metric]
        
                # Process and display results for FOLD-RM for this metric
                fold_rm_df = metric_df[metric_df['fit_method'] == 'FOLD-RM (fit)'].copy()
                fold_rm_filename = f"fold_rm_strategy_summary_{metric}_4.csv"
                process_and_display_results(fold_rm_df, f"FOLD-RM (fit) - Metric: {metric}", fold_rm_filename)
        
                # Process and display results for CON-FOLD for this metric
                con_fold_df = metric_df[metric_df['fit_method'] == 'CON-FOLD (confidence_fit)'].copy()
                con_fold_filename = f"con_fold_strategy_summary_{metric}_4.csv"
                process_and_display_results(con_fold_df, f"CON-FOLD (confidence_fit) - Metric: {metric}", con_fold_filename)
        else:
            print("--- No combinations are fully complete yet (>= " + str(NUM_TRIALS) + " trials). ---")
    else:
        print("--- No trial data found in the database. ---")


>>> Found 0 combinations with remaining trials. <<<


--- No remaining trials to run. Everything is complete. ---
Backup of all completed trial data saved to strategy_experiment_4_30trials_results_full.csv


################################################################################
### PROCESSING RESULTS FOR METRIC: original ###
################################################################################

--- STRATEGY EXPERIMENT SUMMARY: FOLD-RM (fit) - Metric: original ---


No completed data to process for CON-FOLD (confidence_fit) - Metric: original.


################################################################################
### PROCESSING RESULTS FOR METRIC: original_guarded ###
################################################################################

--- STRATEGY EXPERIMENT SUMMARY: FOLD-RM (fit) - Metric: original_guarded ---


No completed data to process for CON-FOLD (confidence_fit) - Metric: original_guarded.


################################################################################
### PROCESSING RESULTS FOR METRIC: original_reweighted ###
################################################################################

--- STRATEGY EXPERIMENT SUMMARY: FOLD-RM (fit) - Metric: original_reweighted ---


No completed data to process for CON-FOLD (confidence_fit) - Metric: original_reweighted.


################################################################################
### PROCESSING RESULTS FOR METRIC: original_t_removed ###
################################################################################

--- STRATEGY EXPERIMENT SUMMARY: FOLD-RM (fit) - Metric: original_t_removed ---


No completed data to process for CON-FOLD (confidence_fit) - Metric: original_t_removed.


################################################################################
### PROCESSING RESULTS FOR METRIC: IG_over_P ###
################################################################################

--- STRATEGY EXPERIMENT SUMMARY: FOLD-RM (fit) - Metric: IG_over_P ---


...

No completed data to process for CON-FOLD (confidence_fit) - Metric: MathewsCC.


################################################################################
### PROCESSING RESULTS FOR METRIC: MathewsCC_mod ###
################################################################################

--- STRATEGY EXPERIMENT SUMMARY: FOLD-RM (fit) - Metric: MathewsCC_mod ---


No completed data to process for CON-FOLD (confidence_fit) - Metric: MathewsCC_mod.


################################################################################
### PROCESSING RESULTS FOR METRIC: MathewsP ###
################################################################################

--- STRATEGY EXPERIMENT SUMMARY: FOLD-RM (fit) - Metric: MathewsP ---


No completed data to process for CON-FOLD (confidence_fit) - Metric: MathewsP.


################################################################################
### PROCESSING RESULTS FOR METRIC: Gini_Impurity ###
################################################################################

--- STRATEGY EXPERIMENT SUMMARY: FOLD-RM (fit) - Metric: Gini_Impurity ---


No completed data to process for CON-FOLD (confidence_fit) - Metric: Gini_Impurity.


################################################################################
### PROCESSING RESULTS FOR METRIC: Precision_Gini_Impurity ###
################################################################################

--- STRATEGY EXPERIMENT SUMMARY: FOLD-RM (fit) - Metric: Precision_Gini_Impurity ---


No completed data to process for CON-FOLD (confidence_fit) - Metric: Precision_Gini_Impurity.


In [3]:
import datasets
import inspect

# 1. Get a list of all functions defined specifically in datasets.py
# (This filters out imported things like 'Classifier' or 'np')
all_functions = [
    func for name, func in inspect.getmembers(datasets, inspect.isfunction)
    if func.__module__ == datasets.__name__
]

print(f"Found {len(all_functions)} datasets. Running checks...\n")

# 2. Loop through them and run
for func in all_functions:
    try:
        # The print statement is already inside your functions, 
        # so we just need to call them.
        func() 
    except FileNotFoundError:
        print(f"!! {func.__name__}: File not found (Data missing)")
    except Exception as e:
        print(f"!! {func.__name__}: Error - {e}")

print("\n--- Finished ---")

Found 35 datasets. Running checks...


% acute dataset (120, 7)

% adult dataset (32561, 15)

% anneal dataset train (798, 39) test (100, 39)

% autism dataset (704, 18)

% avila dataset train (10430, 11) test (10437, 11)

% birds dataset loaded (20, 3)

% breastw dataset (699, 10)

% cars dataset (1728, 7)

% credit dataset (690, 16)

% credit card dataset (30000, 24)

% drug consumption dataset (1885, 13)

% dry bean dataset (13611, 17)

% ecoli dataset (336, 9)

% eeg dataset (14980, 15)

% flags dataset (194, 30)

% glass dataset (214, 10)

% heart dataset (270, 14)

% rain dataset (10459, 24)

% online shoppers intention dataset (12330, 18)

% ionosphere dataset (351, 35)

% kidney dataset (400, 25)

% krkp dataset (3196, 37)

% mushroom dataset (8124, 23)

% nursery dataset (12960, 9)

% page blocks dataset (5473, 11)

% parkison disease dataset (756, 754)

% pendigits train dataset (7494, 17) test (3498, 17)

% rain dataset (145460, 24)

% sonar dataset (208, 61)

% titanic data

In [ ]:
## For multiclass

# Cell 1: Imports and Setup
import numpy as np
import pandas as pd
import time
import os
import sqlite3
from statistics import mean, stdev
from joblib import Parallel, delayed
from IPython.display import display

# Import your project files
from algo import metric_list
from foldrm import Classifier
from utils import split_data, split_xy, get_scores, count_rules_in_model, num_predicates, get_inverse_brier_score
from datasets import wine, cars, glass, flags, ecoli, sonar, weight_lifting

# --- Experiment Configuration ---
# We focus on Wine as requested, but list others for future use
datasets = [wine] 
dataset_names = ["Wine"]

# The comparison we want to make
split_modes = ["Binary", "Multi-class"] 

# Strategies to test
selection_strategies = ['greedy', 'round_robin', 'best_rule', 'info_gain']

# Fixed configuration for this specific experiment
FIT_METHOD = "FOLD-RM (fit)"
METRIC = 'original' # Maps to standard gain in Binary, and Ratio metric in Multi-class
NUM_TRIALS = 30 
DB_FILE = 'multiclass_vs_binary_experiment.db'

# Display options
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Cell 2: Helper Function for a Single Trial
def run_one_trial(dataset_name, strategy, split_mode, data, model_template, num_classes, db_file):
    """
    Runs a single trial. 
    split_mode: "Binary" or "Multi-class"
    """
    start_time = time.time()
    
    # Determine boolean flag for the algorithm
    use_multiclass = True if split_mode == "Multi-class" else False
    
    model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
    data_train, data_test = split_data(data, ratio=0.8)
    X_test, Y_test = split_xy(data_test)
    
    # Pass the multiclass flag to the fit method
    model.fit(data_train, 
              metric=METRIC, 
              num_classes=num_classes, 
              selection_strategy=strategy, 
              multiclass=use_multiclass)
    
    Ystar_test_tuples = model.predict(X_test)
    Ystar_test = [y[0] for y in Ystar_test_tuples]
    score = get_scores(Ystar_test, data_test)
    brier_score = get_inverse_brier_score(Ystar_test_tuples, Y_test)
    
    rule_count = count_rules_in_model(model)
    
    # Calculate predicates (optional, can be slow on complex models)
    model.asp()
    predicate_count = num_predicates(model)
    
    elapsed_time = time.time() - start_time
    
    # Save to database
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute('PRAGMA journal_mode=WAL;')
    cursor.execute('''
        INSERT INTO trials (dataset, strategy, split_mode, metric, accuracy, brier_score, time, num_rules, num_preds)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (dataset_name, strategy, split_mode, METRIC, score, brier_score, elapsed_time, rule_count, predicate_count))
    conn.commit()
    conn.close()

def run_experiments_for_combination(dataset_info, strategy, split_mode, remaining_trials, db_file):
    """
    Runs the remaining trials in parallel.
    """
    dataset_func, name = dataset_info
    print(f"--- Starting: Dataset='{name}', Strategy='{strategy}', Mode='{split_mode}', Remaining={remaining_trials} ---")
    
    model_template, data = dataset_func()
    num_classes = len(pd.unique(pd.DataFrame(data).iloc[:, -1]))
    
    if remaining_trials > 0:
        Parallel(n_jobs=-1, verbose=0)(
            delayed(run_one_trial)(name, strategy, split_mode, data, model_template, num_classes, db_file)
            for _ in range(remaining_trials)
        )
    
    print(f"--- Finished: Dataset='{name}', Strategy='{strategy}', Mode='{split_mode}' ---")

# Cell 3: Main Experiment Runner
if __name__ == "__main__":
    datasets_with_names = list(zip(datasets, dataset_names))

    # Create database
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS trials (
            dataset TEXT,
            strategy TEXT,
            split_mode TEXT,
            metric TEXT,
            accuracy REAL,
            brier_score REAL,
            time REAL,
            num_rules INTEGER,
            num_preds INTEGER
        )
    ''')
    conn.commit()

    # Check existing progress
    counts = {}
    cursor.execute('SELECT dataset, strategy, split_mode, COUNT(*) FROM trials WHERE metric=? GROUP BY dataset, strategy, split_mode', (METRIC,))
    for row in cursor.fetchall():
        ds, strat, mode, cnt = row
        counts[(ds, strat, mode)] = cnt
    conn.close()

    # Generate tasks
    tasks_to_run = []
    for dataset_info in datasets_with_names:
        name = dataset_info[1]
        for strategy in selection_strategies:
            for mode in split_modes:
                key = (name, strategy, mode)
                current_trials = counts.get(key, 0)
                remaining = NUM_TRIALS - current_trials
                if remaining > 0:
                    tasks_to_run.append({
                        'dataset_info': dataset_info,
                        'strategy': strategy,
                        'split_mode': mode,
                        'remaining_trials': remaining,
                        'db_file': DB_FILE
                    })

    print(f"\n>>> Found {len(tasks_to_run)} combinations to run. <<<\n")

    # Execute
    if tasks_to_run:
        for task in tasks_to_run:
            run_experiments_for_combination(**task)
        print("\n--- All tasks complete! ---")
    else:
        print("\n--- No remaining trials to run. ---")

# Cell 4: Comparative Analysis (Binary vs Multi-class)
def display_comparative_results(db_file):
    conn = sqlite3.connect(db_file)
    try:
        df = pd.read_sql_query(f"SELECT * FROM trials WHERE metric='{METRIC}'", conn)
    except:
        print("No data found.")
        return
    conn.close()

    if df.empty:
        print("No data to display.")
        return

    # 1. Aggregate Mean and Std
    agg_df = df.groupby(['dataset', 'strategy', 'split_mode']).agg({
        'accuracy': ['mean', 'std'],
        'time': ['mean'],
        'num_rules': ['mean']
    })
    
    # Flatten columns
    agg_df.columns = ['_'.join(col).strip() for col in agg_df.columns.values]
    agg_df.reset_index(inplace=True)

    # 2. Pivot to put Binary and Multi-class side by side
    # Index: Dataset, Strategy
    # Columns: split_mode (Binary, Multi-class)
    pivoted = agg_df.pivot(index=['dataset', 'strategy'], columns='split_mode')
    
    # Flatten the MultiIndex columns created by pivot
    # Resulting cols will look like: ('accuracy_mean', 'Binary'), ('accuracy_mean', 'Multi-class')
    pivoted.columns = [f"{col[0]}_{col[1]}" for col in pivoted.columns]
    
    # 3. Calculate Delta (Multi-class - Binary) for Accuracy
    if 'accuracy_mean_Multi-class' in pivoted.columns and 'accuracy_mean_Binary' in pivoted.columns:
        pivoted['Acc_Delta'] = pivoted['accuracy_mean_Multi-class'] - pivoted['accuracy_mean_Binary']

    # 4. Reorder for readability
    desired_order = []
    base_metrics = ['accuracy_mean', 'accuracy_std', 'time_mean', 'num_rules_mean']
    
    # Interleave columns: Acc_Binary, Acc_Multi, Time_Binary, Time_Multi...
    for base in base_metrics:
        if f"{base}_Binary" in pivoted.columns: desired_order.append(f"{base}_Binary")
        if f"{base}_Multi-class" in pivoted.columns: desired_order.append(f"{base}_Multi-class")
    
    if 'Acc_Delta' in pivoted.columns:
        desired_order.insert(2, 'Acc_Delta') # Put delta right after accuracy columns

    final_df = pivoted[desired_order]

    print(f"\n{'='*40}")
    print(f"--- BINARY vs MULTI-CLASS COMPARISON (Metric: {METRIC}) ---")
    
    # Styling
    def color_delta(val):
        if pd.isna(val): return ''
        color = 'green' if val > 0 else 'red' if val < 0 else 'black'
        return f'color: {color}; font-weight: bold'

    styled = final_df.style.format("{:.4f}", na_rep="-") \
        .applymap(color_delta, subset=['Acc_Delta'] if 'Acc_Delta' in final_df.columns else [])

    display(styled)

if __name__ == "__main__":
    display_comparative_results(DB_FILE)